In [1]:
import os
from dotenv import load_dotenv
import openai

load_dotenv()
print("Versión de openai:", openai.__version__)
print("Clave encontrada:", bool(os.getenv("GEMINI_API_KEY")))

Versión de openai: 3.29.0
Clave encontrada: True


In [2]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

for modelo in client.models.list().data[:15]:
    print(modelo.id)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview


In [9]:
import time

MODEL = "gemini-3.1-flash-lite-preview"

def call_llm(prompt, temperature=0.7, max_tokens=500, reintentos=3):
    for intento in range(reintentos + 1):
        try:
            respuesta = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return respuesta.choices[0].message.content
        except Exception as e:
            texto = str(e)
            temporal = "503" in texto or "429" in texto
            if temporal and intento < reintentos:
                time.sleep(2 * (intento + 1))
                continue
            return f"Error: {e}"


In [10]:
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

print(call_llm(sentiment_prompt_v1))

This message is classified as **Positive Sentiment** (or **Customer Praise/Feedback**).


In [11]:
import time

def run_n(prompt, n, pausa=1):
    resultados = []
    for i in range(n):
        resultados.append(call_llm(prompt))
        time.sleep(pausa)
    return resultados

sentiment_v1_5 = run_n(sentiment_prompt_v1, 5)

for i, r in enumerate(sentiment_v1_5, 1):
    print(i, "|", r.replace("\n", " ")[:120])

1 | This customer message is classified as **Positive Feedback** (or **Sentiment: Positive**).
2 | This message is classified as **Positive Sentiment** (or **Customer Praise/Feedback**).
3 | The customer message is classified as **Positive Sentiment / Feedback**.
4 | This message is classified as **Positive** (or **Sentiment: Positive**). It expresses customer satisfaction and approval
5 | The classification for this message is: **Positive Sentiment / Customer Feedback.**


In [12]:
from concurrent.futures import ThreadPoolExecutor
import time

def run_n_par(prompt, n, workers=5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda _: call_llm(prompt), range(n)))

t0 = time.time()
mas = run_n_par(sentiment_prompt_v1, 10)
print("Tiempo:", round(time.time() - t0, 1), "s")

sentiment_v1_15 = sentiment_v1_5 + mas
errores = sum(r.startswith("Error") for r in sentiment_v1_15)
print(len(sentiment_v1_15), "respuestas;", errores, "errores")

Tiempo: 44.1 s
15 respuestas; 1 errores


In [13]:
def es_valida(r):
    return r.strip().strip(".").lower() in {"positive", "negative", "neutral"}

for i, r in enumerate(sentiment_v1_15, 1):
    if r.startswith("Error"):
        print(i, "ERROR:", r[:300])

reales = [r for r in sentiment_v1_15 if not r.startswith("Error")]
validas = [r for r in reales if es_valida(r)]

print("Respuestas reales:", len(reales))
print("Válidas con el criterio:", len(validas))
print("Respuestas distintas:", len(set(reales)))

15 ERROR: Error: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Respuestas reales: 14
Válidas con el criterio: 0
Respuestas distintas: 10


In [6]:
candidatos = [
    "gemini-2.5-flash-lite",
    "gemini-flash-latest",
    "gemini-flash-lite-latest",
    "gemini-3-flash-preview",
    "gemini-3.1-flash-lite-preview",
]

for nombre in candidatos:
    MODEL = nombre
    respuesta = call_llm("Say hi in one word.", max_tokens=300)
    ok = not str(respuesta).startswith("Error")
    print(nombre, "->", "OK" if ok else "falla", "|", str(respuesta)[:60])

gemini-2.5-flash-lite -> falla | Error: Error code: 404 - [{'error': {'code': 404, 'message':
gemini-flash-latest -> falla | Error: Error code: 503 - [{'error': {'code': 503, 'message':
gemini-flash-lite-latest -> OK | Hi.
gemini-3-flash-preview -> OK | Hi.
gemini-3.1-flash-lite-preview -> OK | Hi.


In [1]:
print(len(sentiment_v1_15))

NameError: name 'sentiment_v1_15' is not defined

In [2]:
import os, time
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

def call_llm(prompt, temperature=0.7, max_tokens=500, reintentos=3):
    for intento in range(reintentos + 1):
        try:
            respuesta = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return respuesta.choices[0].message.content
        except Exception as e:
            texto = str(e)
            temporal = "503" in texto or "429" in texto
            if temporal and intento < reintentos:
                time.sleep(2 * (intento + 1))
                continue
            return f"Error: {e}"

def run_n_par(prompt, n, workers=5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda _: call_llm(prompt), range(n)))

def es_valida_sentimiento(r):
    return r.strip().strip(".").lower() in {"positive", "negative", "neutral"}

print("Modelo:", MODEL)
print("Clave encontrada:", bool(os.getenv("GEMINI_API_KEY")))

Modelo: gemini-3.1-flash-lite-preview
Clave encontrada: True


In [3]:
import json, datetime

os.makedirs("resultados", exist_ok=True)

sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

def guardar_serie(nombre, prompt, respuestas, temperature=0.7):
    datos = {
        "nombre": nombre,
        "modelo": MODEL,
        "temperature": temperature,
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "prompt": prompt,
        "respuestas": respuestas,
    }
    with open(f"resultados/{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

t0 = time.time()
sentiment_v1_15 = run_n_par(sentiment_prompt_v1, 15)
product_v1_15 = run_n_par(product_prompt_v1, 15)
extraction_v1_15 = run_n_par(extraction_prompt_v1, 15)
print("Tiempo:", round(time.time() - t0, 1), "s")

guardar_serie("sentimiento_v1", sentiment_prompt_v1, sentiment_v1_15)
guardar_serie("producto_v1", product_prompt_v1, product_v1_15)
guardar_serie("extraccion_v1", extraction_prompt_v1, extraction_v1_15)

for nombre, serie in [("sentimiento", sentiment_v1_15), ("producto", product_v1_15), ("extracción", extraction_v1_15)]:
    errores = sum(r.startswith("Error") for r in serie)
    print(nombre, "-", len(serie), "respuestas;", errores, "errores")

Tiempo: 136.9 s
sentimiento - 15 respuestas; 3 errores
producto - 15 respuestas; 3 errores
extracción - 15 respuestas; 0 errores


In [6]:
series = [
    ("sentimiento", sentiment_v1_15),
    ("producto", product_v1_15),
    ("extracción", extraction_v1_15),
]

for nombre, serie in series:
    print(nombre, "- errores:", [r[:45] for r in serie if r.startswith("Error")])

reales_s = [r for r in sentiment_v1_15 if not r.startswith("Error")]
print()
print("SENTIMIENTO: reales", len(reales_s),
      "| válidas", sum(es_valida_sentimiento(r) for r in reales_s),
      "| distintas", len(set(reales_s)))
for r in reales_s:
    print("-", r.replace("\n", " ")[:110])

reales_p = [r for r in product_v1_15 if not r.startswith("Error")]
print()
print("PRODUCTO: palabras por respuesta y primeras palabras")
for r in reales_p:
    print(len(r.split()), "palabras |", r.replace("\n", " ")[:90])

reales_e = [r for r in extraction_v1_15 if not r.startswith("Error")]
print()
print("EXTRACCIÓN: primeras palabras")
for r in reales_e:
    print("-", r.replace("\n", " ")[:140])

sentimiento - errores: ["Error: Error code: 429 - [{'error': {'code': ", "Error: Error code: 429 - [{'error': {'code': ", "Error: Error code: 429 - [{'error': {'code': "]
producto - errores: ["Error: Error code: 429 - [{'error': {'code': ", "Error: Error code: 429 - [{'error': {'code': ", "Error: Error code: 429 - [{'error': {'code': "]
extracción - errores: []

SENTIMIENTO: reales 12 | válidas 0 | distintas 10
- This message can be classified as:  *   **Sentiment:** Positive *   **Category:** Feedback / Testimonial *   *
- This message is classified as **Positive** (or **Sentiment: Positive**). It expresses customer satisfaction an
- This message can be classified as:  *   **Sentiment:** Positive *   **Category:** Customer Feedback / Testimon
- Classification: **Positive / Positive Feedback**
- The customer message can be classified as:  *   **Sentiment:** Positive *   **Category:** Feedback / Praise * 
- Classification: **Positive / Compliment**
- This customer message can be classif

In [7]:
with open("resultados/lectura_v1.txt", "w", encoding="utf-8") as f:
    for nombre, serie in series:
        f.write(f"===== {nombre.upper()} v1 =====\n\n")
        for i, r in enumerate(serie, 1):
            f.write(f"--- Respuesta {i} ---\n{r}\n\n")
print("Archivo creado: resultados/lectura_v1.txt")

Archivo creado: resultados/lectura_v1.txt


In [8]:
reales_p = [r for r in product_v1_15 if not r.startswith("Error")]

palabras = ["2.4", "whisper", "dpi", "battery", "bluetooth", "ergonomic", "warranty"]

print("Respuestas de producto reales:", len(reales_p))
for p in palabras:
    n = sum(p in r.lower() for r in reales_p)
    print(f"'{p}' aparece en {n} de {len(reales_p)}")

con_alguna = sum(any(p in r.lower() for p in palabras) for r in reales_p)
print("Con al menos una de esas palabras:", con_alguna, "de", len(reales_p))

Respuestas de producto reales: 12
'2.4' aparece en 11 de 12
'whisper' aparece en 11 de 12
'dpi' aparece en 9 de 12
'battery' aparece en 12 de 12
'bluetooth' aparece en 0 de 12
'ergonomic' aparece en 12 de 12
'warranty' aparece en 0 de 12
Con al menos una de esas palabras: 12 de 12


## Análisis de fallos de los prompts v1 (Parte 2)

Modelo `gemini-3.1-flash-lite-preview`, temperature 0.7. Serie de 15 ejecuciones por prompt. Los errores 503 del servidor no cuentan como respuestas del modelo. Detalle completo en `analisis_fallos_v1.md`.

| Prompt v1 | Errores 503 | Respuestas reales | Distintas | Fallo principal |
|---|---|---|---|---|
| Sentimiento | 3 | 12 | 10 | Sentimiento correcto (Positive en 12 de 12), pero ninguna respuesta es una sola palabra: 0 de 12 válidas |
| Producto | 3 | 12 | 12 | Cada respuesta da tres opciones de descripción (293 a 325 palabras) e inventa especificaciones: "2.4" en 11 de 12, "whisper" en 11 de 12, "dpi" en 9 de 12 |
| Extracción | 0 | 15 | 1 | Ninguno: las 15 respuestas son idénticas y contienen los 4 datos |

**Hipótesis:** cuantos más huecos deja el prompt, más rellena el modelo y más varían las respuestas. En extracción los datos ya están en el texto y el modelo los copia.

## Iteración 1 (v2): sentimiento

Cambios respecto a v1 (propuesta aprobada): opciones cerradas (Positive, Negative, Neutral), una sola palabra, sin explicación ni puntuación. Criterio de respuesta válida: la respuesta, sin espacios ni punto final, es exactamente una de las tres opciones.

In [9]:
sentiment_prompt_v2 = """Classify the sentiment of this customer message as exactly one word: Positive, Negative, or Neutral. Reply with only that word, with no explanation and no punctuation.

Customer message: "I love this product! It's exactly what I needed."
"""

sentiment_v2_15 = run_n_par(sentiment_prompt_v2, 15)
guardar_serie("sentimiento_v2", sentiment_prompt_v2, sentiment_v2_15)

reales_s2 = [r for r in sentiment_v2_15 if not r.startswith("Error")]
validas_s2 = [r for r in reales_s2 if es_valida_sentimiento(r)]

print("Errores:", len(sentiment_v2_15) - len(reales_s2))
print("Respuestas reales:", len(reales_s2))
print("Válidas:", len(validas_s2), "| distintas:", len(set(reales_s2)))
if reales_s2:
    print("Consistencia v2:", round(100 * len(validas_s2) / len(reales_s2), 1), "%")
for r in reales_s2:
    print("-", repr(r))

Errores: 0
Respuestas reales: 15
Válidas: 15 | distintas: 1
Consistencia v2: 100.0 %
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'


## Iteración 1 (v2): producto

Cambios respecto a v1 (propuesta aprobada): una sola descripción en un párrafo de 40 a 60 palabras, sin título ni viñetas, usando solo los datos dados y sin inventar especificaciones. Criterio de respuesta válida: entre 40 y 60 palabras, un solo párrafo sin viñetas, y ningún número aparte de 29.99.

In [10]:
import re

product_prompt_v2 = """Write a product description for a wireless mouse that costs $29.99. Write one single paragraph of 40 to 60 words, with no title and no bullet points. Use only the facts given here. Do not invent specifications, numbers or features such as connectivity technology, DPI, battery life or warranty."""

def evaluar_producto(r):
    t = r.strip()
    return {
        "palabras_40_60": 40 <= len(t.split()) <= 60,
        "un_parrafo": "\n" not in t,
        "sin_vinetas": not re.search(r"^\s*([*\-•#]|\d+\.)\s", t, re.M),
        "sin_numeros_extra": not re.search(r"\d", t.replace("29.99", "")),
    }

def es_valida_producto(r):
    return all(evaluar_producto(r).values())

product_v2_15 = run_n_par(product_prompt_v2, 15)
guardar_serie("producto_v2", product_prompt_v2, product_v2_15)

reales_p2 = [r for r in product_v2_15 if not r.startswith("Error")]
validas_p2 = [r for r in reales_p2 if es_valida_producto(r)]

print("Errores:", len(product_v2_15) - len(reales_p2))
print("Respuestas reales:", len(reales_p2))
print("Válidas:", len(validas_p2), "| distintas:", len(set(reales_p2)))
if reales_p2:
    print("Consistencia v2:", round(100 * len(validas_p2) / len(reales_p2), 1), "%")
    for criterio in ["palabras_40_60", "un_parrafo", "sin_vinetas", "sin_numeros_extra"]:
        cumplen = sum(evaluar_producto(r)[criterio] for r in reales_p2)
        print(f"  {criterio}: {cumplen} de {len(reales_p2)}")
print()
for r in reales_p2:
    print(len(r.split()), "palabras |", r.replace("\n", " ")[:160])

Errores: 0
Respuestas reales: 15
Válidas: 15 | distintas: 15
Consistencia v2: 100.0 %
  palabras_40_60: 15 de 15
  un_parrafo: 15 de 15
  sin_vinetas: 15 de 15
  sin_numeros_extra: 15 de 15

51 palabras | This wireless mouse provides a reliable and convenient solution for your daily computing needs. Priced at just $29.99, it offers a functional design that enhanc
51 palabras | Experience reliable performance with this sleek wireless mouse, available now for only $29.99. Designed for comfort and efficiency, this peripheral offers a sim
53 palabras | Experience smooth navigation and reliable performance with this versatile wireless mouse. Designed for comfort and efficiency, it provides a seamless user exper
53 palabras | Experience effortless navigation with this reliable wireless mouse, available now for only $29.99. Designed for comfort and everyday efficiency, this essential 
52 palabras | Experience comfort and convenience with this high-quality wireless mouse. Designed for efficien

## Iteración 1 (v2): extracción

Cambios respecto a v1 (propuesta aprobada): JSON con cuatro claves fijas (order_number, order_date, delivery, packaging), solo información del texto, sin explicación ni bloque de código. Criterio de respuesta válida: el JSON se lee sin errores, tiene exactamente esas cuatro claves, order_number contiene 12345 y order_date contiene March 15. Si delivery y packaging recogen lo que dice el texto se revisa a mano.

In [11]:
import json

extraction_prompt_v2 = """Extract the following fields from this customer feedback and return them as JSON with exactly these four keys: order_number, order_date, delivery, packaging. Use only information from the text. Return only the JSON, with no explanation and no code block.

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

CLAVES = {"order_number", "order_date", "delivery", "packaging"}

def evaluar_extraccion(r):
    try:
        d = json.loads(r.strip())
    except Exception:
        return {"json_valido": False, "cuatro_claves": False, "pedido_ok": False, "fecha_ok": False}
    ok_dict = isinstance(d, dict)
    return {
        "json_valido": True,
        "cuatro_claves": ok_dict and set(d.keys()) == CLAVES,
        "pedido_ok": ok_dict and "12345" in str(d.get("order_number", "")),
        "fecha_ok": ok_dict and "March 15" in str(d.get("order_date", "")),
    }

def es_valida_extraccion(r):
    return all(evaluar_extraccion(r).values())

extraction_v2_15 = run_n_par(extraction_prompt_v2, 15)
guardar_serie("extraccion_v2", extraction_prompt_v2, extraction_v2_15)

reales_e2 = [r for r in extraction_v2_15 if not r.startswith("Error")]
validas_e2 = [r for r in reales_e2 if es_valida_extraccion(r)]

print("Errores:", len(extraction_v2_15) - len(reales_e2))
print("Respuestas reales:", len(reales_e2))
print("Válidas:", len(validas_e2), "| distintas:", len(set(reales_e2)))
if reales_e2:
    print("Consistencia v2:", round(100 * len(validas_e2) / len(reales_e2), 1), "%")
    for criterio in ["json_valido", "cuatro_claves", "pedido_ok", "fecha_ok"]:
        cumplen = sum(evaluar_extraccion(r)[criterio] for r in reales_e2)
        print(f"  {criterio}: {cumplen} de {len(reales_e2)}")
print()
for r in reales_e2:
    print("-", r.replace("\n", " ")[:200])

Errores: 0
Respuestas reales: 15
Válidas: 15 | distintas: 1
Consistencia v2: 100.0 %
  json_valido: 15 de 15
  cuatro_claves: 15 de 15
  pedido_ok: 15 de 15
  fecha_ok: 15 de 15

- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "order_number": "12345", "order_date": "March 15th", "delivery": "fast", "packaging": "damaged" }
- { "

## Preparación para la v3 (sin llamadas a Gemini)

Esta celda vuelve a cargar la configuración y las funciones de evaluación. No hace ninguna llamada al modelo. Sirve cuando se reinicia el kernel (el cuaderno se cierra y se vuelve a abrir). Ejecutarla una vez antes de las celdas v3.

In [12]:
import os, re, json, time, datetime
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

def call_llm(prompt, temperature=0.7, max_tokens=500, reintentos=3):
    for intento in range(reintentos + 1):
        try:
            respuesta = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return respuesta.choices[0].message.content
        except Exception as e:
            texto = str(e)
            temporal = "503" in texto or "429" in texto
            if temporal and intento < reintentos:
                time.sleep(2 * (intento + 1))
                continue
            return f"Error: {e}"

def run_n_par(prompt, n, workers=5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda _: call_llm(prompt), range(n)))

os.makedirs("resultados", exist_ok=True)

def guardar_serie(nombre, prompt, respuestas, temperature=0.7):
    datos = {
        "nombre": nombre,
        "modelo": MODEL,
        "temperature": temperature,
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "prompt": prompt,
        "respuestas": respuestas,
    }
    with open(f"resultados/{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

def es_valida_sentimiento(r):
    return r.strip().strip(".").lower() in {"positive", "negative", "neutral"}

def evaluar_producto(r):
    t = r.strip()
    return {
        "palabras_40_60": 40 <= len(t.split()) <= 60,
        "un_parrafo": "\n" not in t,
        "sin_vinetas": not re.search(r"^\s*([*\-•#]|\d+\.)\s", t, re.M),
        "sin_numeros_extra": not re.search(r"\d", t.replace("29.99", "")),
    }

def es_valida_producto(r):
    return all(evaluar_producto(r).values())

CLAVES = {"order_number", "order_date", "delivery", "packaging"}

def evaluar_extraccion(r):
    try:
        d = json.loads(r.strip())
    except Exception:
        return {"json_valido": False, "cuatro_claves": False, "pedido_ok": False, "fecha_ok": False}
    ok_dict = isinstance(d, dict)
    return {
        "json_valido": True,
        "cuatro_claves": ok_dict and set(d.keys()) == CLAVES,
        "pedido_ok": ok_dict and "12345" in str(d.get("order_number", "")),
        "fecha_ok": ok_dict and "March 15" in str(d.get("order_date", "")),
    }

def es_valida_extraccion(r):
    return all(evaluar_extraccion(r).values())

print("Modelo:", MODEL)
print("Clave encontrada:", bool(os.getenv("GEMINI_API_KEY")))
print("Preparación lista: funciones cargadas, ninguna llamada al modelo.")

Modelo: gemini-3.1-flash-lite-preview
Clave encontrada: True
Preparación lista: funciones cargadas, ninguna llamada al modelo.


## Iteración 2 (v3): sentimiento con ejemplos (few-shot)

Propuesta de la IA, a revisar por Jaime. Se mantiene la instrucción de v2 y se añaden tres ejemplos de sentimientos distintos (positivo, negativo, neutro), basados en las ideas del borrador de Jaime (paquete roto, llegada puntual), reescritos como mensajes de cliente. Se cambia solo una cosa respecto a v2 (los ejemplos) para poder atribuir el efecto.

In [13]:
sentiment_prompt_v3 = """Classify the sentiment of each customer message as exactly one word: Positive, Negative, or Neutral. Reply with only that word, with no explanation and no punctuation.

Message: "The package arrived on time and everything works perfectly."
Sentiment: Positive

Message: "The box was torn and nobody has answered my complaint."
Sentiment: Negative

Message: "I ordered the blue model on Tuesday."
Sentiment: Neutral

Message: "I love this product! It's exactly what I needed."
Sentiment:"""

sentiment_v3_15 = run_n_par(sentiment_prompt_v3, 15)
guardar_serie("sentimiento_v3", sentiment_prompt_v3, sentiment_v3_15)

reales_s3 = [r for r in sentiment_v3_15 if not r.startswith("Error")]
validas_s3 = [r for r in reales_s3 if es_valida_sentimiento(r)]
print("Errores:", len(sentiment_v3_15) - len(reales_s3))
print("Respuestas reales:", len(reales_s3))
print("Válidas:", len(validas_s3), "| distintas:", len(set(reales_s3)))
if reales_s3:
    print("Consistencia v3:", round(100 * len(validas_s3) / len(reales_s3), 1), "%")
for r in reales_s3:
    print("-", repr(r))

Errores: 0
Respuestas reales: 15
Válidas: 15 | distintas: 1
Consistencia v3: 100.0 %
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'
- 'Positive'


## Iteración 2 (v3): extracción con razonamiento paso a paso (Chain-of-Thought)

Propuesta de la IA, a revisar por Jaime. Se mantienen los cuatro campos de v2 y se pide razonar antes: primero pedido y fecha, luego entrega, luego embalaje. El resultado final va tras la marca fija `FINAL JSON:` (idea de Jaime). Criterio de respuesta válida: contiene la marca, y lo que viene detrás es un JSON con las cuatro claves, con pedido 12345 y fecha March 15.

In [14]:
extraction_prompt_v3 = """Extract the following fields from this customer feedback: order_number, order_date, delivery, packaging.

First think step by step: find the order number and the date, then find what the customer says about the delivery, then what the customer says about the packaging. Use only information from the text.

After your reasoning, write the line "FINAL JSON:" followed by a JSON object with exactly these four keys, and nothing after it.

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

def json_final(r):
    if "FINAL JSON:" not in r:
        return None
    parte = r.split("FINAL JSON:")[-1].strip()
    if parte.startswith("```"):
        parte = parte.strip("`").strip()
        if parte.lower().startswith("json"):
            parte = parte[4:].strip()
    return parte

def evaluar_extraccion_v3(r):
    parte = json_final(r)
    res = {"tiene_marca": parte is not None}
    res.update(evaluar_extraccion(parte if parte is not None else ""))
    return res

extraction_v3_15 = run_n_par(extraction_prompt_v3, 15)
guardar_serie("extraccion_v3", extraction_prompt_v3, extraction_v3_15)

reales_e3 = [r for r in extraction_v3_15 if not r.startswith("Error")]
validas_e3 = [r for r in reales_e3 if all(evaluar_extraccion_v3(r).values())]
print("Errores:", len(extraction_v3_15) - len(reales_e3))
print("Respuestas reales:", len(reales_e3))
print("Válidas:", len(validas_e3))
if reales_e3:
    print("Consistencia v3:", round(100 * len(validas_e3) / len(reales_e3), 1), "%")
    for criterio in ["tiene_marca", "json_valido", "cuatro_claves", "pedido_ok", "fecha_ok"]:
        print(f"  {criterio}:", sum(evaluar_extraccion_v3(r)[criterio] for r in reales_e3), "de", len(reales_e3))
print()
print("Primera respuesta completa, para ver el razonamiento:")
print(reales_e3[0] if reales_e3 else "sin respuestas")

Errores: 4
Respuestas reales: 11
Válidas: 11
Consistencia v3: 100.0 %
  tiene_marca: 11 de 11
  json_valido: 11 de 11
  cuatro_claves: 11 de 11
  pedido_ok: 11 de 11
  fecha_ok: 11 de 11

Primera respuesta completa, para ver el razonamiento:
Step 1: Identify the order number. The text states "item #12345", so the order_number is "12345".
Step 2: Identify the order date. The text states "March 15th", so the order_date is "March 15th".
Step 3: Identify the feedback regarding delivery. The text states "The delivery was fast", so delivery is "fast".
Step 4: Identify the feedback regarding packaging. The text states "the packaging was damaged", so packaging is "damaged".

FINAL JSON:
{
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery": "fast",
  "packaging": "damaged"
}


## Iteración 2 (v3): producto con ejemplo y estructura

Propuesta de la IA, a revisar por Jaime. Se mantiene la estructura de v2 (un párrafo, 40 a 60 palabras, sin título ni viñetas, solo datos dados) y se añade un ejemplo con otro producto (mochila, idea de Jaime). El ejemplo cumple las mismas reglas: 49 palabras y ninguna cifra aparte del precio.

In [15]:
product_prompt_v3 = """Write a product description for the product below. Write one single paragraph of 40 to 60 words, with no title and no bullet points. Use only the facts given: the product type and the price. Do not invent specifications, numbers or features.

Product: backpack, $49.99
Description: This backpack offers a practical way to carry your daily essentials, from work to weekend plans. Its straightforward design keeps things simple and organized, making it an easy companion for commuting, studying or travelling. At $49.99, it is an accessible choice for anyone looking for a reliable, no-fuss bag.

Product: wireless mouse, $29.99
Description:"""

product_v3_15 = run_n_par(product_prompt_v3, 15)
guardar_serie("producto_v3", product_prompt_v3, product_v3_15)

reales_p3 = [r for r in product_v3_15 if not r.startswith("Error")]
validas_p3 = [r for r in reales_p3 if es_valida_producto(r)]
print("Errores:", len(product_v3_15) - len(reales_p3))
print("Respuestas reales:", len(reales_p3))
print("Válidas:", len(validas_p3), "| distintas:", len(set(reales_p3)))
if reales_p3:
    print("Consistencia v3:", round(100 * len(validas_p3) / len(reales_p3), 1), "%")
    for criterio in ["palabras_40_60", "un_parrafo", "sin_vinetas", "sin_numeros_extra"]:
        print(f"  {criterio}:", sum(evaluar_producto(r)[criterio] for r in reales_p3), "de", len(reales_p3))
print()
for r in reales_p3[:5]:
    print(len(r.split()), "palabras |", r.replace("\n", " ")[:160])

Errores: 0
Respuestas reales: 15
Válidas: 15 | distintas: 15
Consistencia v3: 100.0 %
  palabras_40_60: 15 de 15
  un_parrafo: 15 de 15
  sin_vinetas: 15 de 15
  sin_numeros_extra: 15 de 15

53 palabras | This wireless mouse is an accessible choice for anyone needing a reliable tool for their computer setup. Designed for convenience and ease of use, it provides a
52 palabras | This wireless mouse is an efficient and functional tool designed to help you navigate your digital workspace with ease. Priced at $29.99, it serves as a practic
52 palabras | This wireless mouse is an efficient tool for your daily computing needs, offering a convenient way to navigate your tasks with ease. Whether you are working at 
52 palabras | This wireless mouse is an essential accessory for your computer setup, offering a practical solution for navigating your digital tasks with ease. Designed for c
54 palabras | This wireless mouse is an accessible choice for anyone needing a reliable tool for their comput